# INSTRUCTIONS: 
Please execute the cells in the notebook as isntructed. You will have to complete some TODO sections in order to proceed further and other cells you will just have to RUN.

[IMPORTANT]: PLEASE CHOOSE 'TORCH' KERNEL

## 1. Setup the environment and define utility function (RUN only)

In [ ]:
!pip install ctransformers

In [ ]:
from ctransformers import AutoModelForCausalLM
from datasets import load_dataset

# Load Mistral model
mistral = AutoModelForCausalLM.from_pretrained(
    "TheBloke/Mistral-7B-Instruct-v0.1-GGUF",
    model_file="mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    model_type="mistral",
)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import uuid
import time
from IPython.display import display, HTML, Javascript
import html as html_lib


def generate_mistral_response(prompt):
    """
    Runs the model on a given prompt and returns the response text and elapsed time.
    """
    start = time.time()
    response_text = mistral(prompt)
    elapsed = time.time() - start
    return str(response_text), elapsed

In [ ]:
def display_sample(question, answer):
    display(HTML(
        f"""<p>Question/answer 0:</p>
        <strong>Question: </strong>{question}
        <p><strong>Answer: </strong>{answer}</p>
        """
    ))

def display_mistral(prompt):
    """
    Displays the prompt and model response in a styled table,
    with a loading indicator while the model generates text.
    """
    uid = str(uuid.uuid4()).replace("-", "")

    # HTML scaffold (before generation)
    html = f"""
    <style>
    .prompt-table {{
        border-collapse: collapse;
        width: 100%;
        margin: 12px 0;
        font-family: 'Segoe UI', sans-serif;
    }}
    .prompt-table th, .prompt-table td {{
        border: 1px solid #ccc;
        padding: 10px;
        vertical-align: top;
    }}
    .prompt-table th {{
        background-color: #f2f2f2;
        width: 20%;
    }}
    .loading {{
        color: #888;
        font-style: italic;
        animation: pulse 1.5s infinite;
    }}
    @keyframes pulse {{
        0% {{ opacity: 0.3; }}
        50% {{ opacity: 1; }}
        100% {{ opacity: 0.3; }}
    }}
    </style>
    <table class="prompt-table">
        <tr><th>Prompt</th><td>{prompt}</td></tr>
        <tr><th>Model Response</th><td id="response_{uid}">
            <span class="loading">⏳ Generating response...</span>
        </td></tr>
    </table>
    """

    # Display initial table
    display(HTML(html))

    # --- Call the model separately ---
    response_text, elapsed = generate_mistral_response(prompt)
    # print(response_text)
    # Escape special chars for HTML display
    safe_response = html_lib.escape(response_text.strip())

    # --- Inject response dynamically ---
    js = Javascript(f"""
        document.getElementById("response_{uid}").innerHTML =
            `<pre style="white-space: pre-wrap;">{safe_response}</pre>
             <div style='color:#666; font-size:90%; margin-top:4px;'>⏱ Generated in {elapsed:.2f} seconds</div>`;
    """)
    display(js)


## 2. Chain of Thought

In this exercise, we’ll explore how the way we prompt a model dramatically changes its reasoning ability. Using a problem from the GSM8K math dataset (The creators of this dataset have already provided the train and test split so we will save it accordingly into the variables), we’ll start with a simple, direct question, letting the model respond without any guidance. You’ll likely see a quick but shallow answer (sometimes correct, often not) showing how a model can jump to conclusions without structured thinking.

Next, we’ll introduce a **Chain-of-Thought (CoT)** prompt, encouraging the model to “think step by step.” This small change transforms the process: instead of guessing, the model begins to reason through intermediate steps, showing its logic and improving accuracy. You’ll see how adding transparency to the model’s thought process helps it solve more complex problems.

By the end, you’ll see how these progressive prompting techniques shift the model from surface-level pattern matching to structured reasoning. The goal is not just to get the right answer, but to understand how the right prompt design unlocks better thinking.

[RUN] Let's start! We can start by exploring our dataset and getting an idea of how the questions and answers are structured.

In [ ]:
ds_main = load_dataset("openai/gsm8k", "main")
train_split_main = ds_main['train']
test_split_main = ds_main['test']

print(f"Train split size: {len(train_split_main)}")
print(f"Test split size: {len(test_split_main)}")


Train split size: 7473
Test split size: 1319


In [ ]:
display_sample(train_split_main[0]['question'], train_split_main[0]['answer'])

[RUN] Now, let's ask our model to solve this question. We will invoke the model by just passing it the first question, and nothing more ( **zero-shot prompting** )

In [ ]:
question = train_split_main[0]['question']
prompt = f"""{question}"""

display_mistral(prompt)

Prompt,"Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
Model Response,⏳ Generating response...


<IPython.core.display.Javascript object>

Hmmm, it was not very helpful, was it? It looks like our model did not reason properly. Let's find a way to trigger the chain of thought process!

### TODO: Try It!
- Complete the `cot_prompt` variable with a piece of prompt that will stimulate the chain-of-thought process in the LLM and make it reason to solve the problem.

In [ ]:
cot_prompt = "" # TODO: Add CoT prompt here
prompt = f"""{cot_prompt}
{question}"""

display_mistral(prompt)

Prompt,"Let's solve this problem step-by-step. Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
Model Response,⏳ Generating response...


<IPython.core.display.Javascript object>

Amazing! Such a simple addition made our model considerably better at a complex task involving reasoning steps!